# 01-3. Tool Calling 기초
- 핵심 기술: Tool Schema, `bind_tools`, Tool Call, Tool Message

## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. Function Calling과 Tool Calling의 관계를 설명할 수 있다.
2. Python 함수, Tool Schema, Tool Call, Tool Message의 차이를 설명할 수 있다.
3. LLM이 제안한 Tool 호출 인자를 검증한 뒤 실제 Python 함수를 실행할 수 있다.
4. Tool 실행 결과를 최종 응답 생성에 반영할 수 있다.
5. 존재하지 않는 Tool 호출이나 잘못된 인자에 대한 오류를 처리할 수 있다.

## 2. 문제 상황

"25와 17을 더해줘"라는 요청을 LLM이 직접 텍스트로 계산해서 답하면, 자릿수가 큰 계산이나 복잡한 수식에서 오차가 발생할 수 있다.  
마찬가지로 "이 문서를 읽어서 알려줘"라는 요청도 LLM 스스로는 로컬 파일에 접근할 수 없다.  
계산 함수, 파일 조회, 웹 검색처럼 LLM의 텍스트 생성 바깥에 있는 기능을 **Tool**로 연결하면 이 한계를 보완할 수 있다.  
이때 LLM은 사용할 Tool과 인자를 **제안**하고, 애플리케이션은 해당 Tool을 **실행**한 뒤 결과를 다시 LLM에게 전달한다. 이 전체 상호작용을 **Tool Calling**이라고 한다.   

> LLM은 Tool 호출을 제안하고 인자를 생성하지만, 실제 Python 함수를 실행하는 주체는
> 애플리케이션이다.

## 3. 핵심 개념

### 3.1 개념 정의

> **용어 안내**: **Function Calling**은 모델이 개발자가 정의한 함수 이름과 인자를  
> 생성하는 방식을 가리킨다. **Tool Calling**은 함수형 Tool뿐 아니라 웹 검색, 파일 검색,  
> MCP 같은 도구 사용까지 포괄하는 더 넓은 표현이다. 프레임워크나 모델 제공자에 따라  
> 두 용어를 혼용하기도 하지만, 이 과정에서는 확장성을 고려해 **Tool Calling**으로  
> 통일한다. 이 실습에서 다루는 Python 함수는 Tool의 한 종류이다.  

**Python 함수**는 일반적인 코드이고, **Tool**은 LLM이 사용할 수 있도록 이름·설명·인자 형식을 노출한 외부 기능이다.   
이 실습에서는 Python 함수를 Tool로 변환한다.  
**Tool Schema**는 Tool의 이름, 설명, 인자 이름과 자료형을 LLM이 이해할 수 있는 형식(JSON Schema)으로 정의한 것이다.   
**Tool Call**은 LLM이 "이 Tool을 이런 인자로 호출하고 싶다"고 제안한 결과이고,   
**Tool Message**는 애플리케이션이 실제로 Tool을 실행한 결과를 다시 LLM에게 전달하는 메시지이다.  

### 3.2 개념이 필요한 이유

LLM은 언어 생성에는 강하지만 정확한 산술 연산이나 로컬 파일 접근, 최신 데이터 조회처럼  
결정적이고 외부 자원에 의존하는 작업에는 취약하다.   
Tool Calling이 없다면 이런 요청마다 LLM이 부정확한 값을 추측해서 답하거나, 아예 처리할 수 없다고 답하는 수밖에 없다.  

### 3.3 주요 구성요소

| 구성요소 | 역할 |
|---|---|
| Tool 이름 | LLM이 사용할 도구를 식별하는 이름 |
| Tool 설명 | 이 Tool이 무엇을 하는지 LLM에게 알려주는 문장 |
| 인자 Schema | 인자 이름과 자료형을 정의 |
| Tool Call | LLM이 제안한 (Tool 이름, 인자) 쌍 |
| Tool 실행기 | Tool Call을 받아 등록된 Tool을 실행하는 애플리케이션 코드 |
| Tool Message | 실행 결과를 LLM에게 되돌려주는 메시지 |



### 3.4 동작 과정

```mermaid
flowchart TD
    A[User 요청] --> B["LLM이 필요한 Tool과 인자를 제안<br/>(Tool Call, AIMessage.tool_calls)"]
    B --> C[애플리케이션이 인자를 검증]
    C --> D["선택된 Tool 실행<br/>(이 실습에서는 Python 함수)"]
    D --> E[실행 결과를 Tool Message로 포장해서 LLM에 전달]
    E --> F[LLM이 Tool Message를 참고하여 최종 응답 생성]
```

### 3.5 코드와 개념의 대응 관계

| 코드 요소 | 구현 개념 |
|---|---|
| `@tool` 데코레이터 | Python 함수를 Tool Schema로 변환 |
| `model.bind_tools(TOOLS)` | LLM에 사용 가능한 Tool 목록 등록 |
| `ai_message.tool_calls` | LLM이 제안한 Tool Call 목록 |
| `execute_tool_call()` | 애플리케이션이 실제 함수를 실행하는 단계 |
| `ToolMessage(...)` | 실행 결과를 LLM에 되돌려주는 메시지 |

### 3.6 Function Calling과 Tool Calling

| 구분 | Function Calling | Tool Calling |
|---|---|---|
| 범위 | 개발자가 정의한 함수 호출 | 함수형 Tool과 검색·파일·MCP 등 다양한 Tool 사용 |
| 공통점 | 모델이 이름과 인자를 제안하고 애플리케이션이 실행 | 모델이 Tool과 인자를 제안하고 실행 환경이 실행 |
| 이 실습에서의 의미 | `calculate_tool` 같은 함수형 Tool | 등록·선택·실행·결과 반환의 전체 과정 |

따라서 이 실습의 구현을 Function Calling이라고 불러도 기술적으로 틀리지는 않지만,  
이후 웹 검색과 RAG, MCP 등으로 확장되는 과정 전체를 설명하기 위해 Tool Calling이라는 용어를 사용한다.  

### 3.7 Tool 선택과 Tool 실행의 차이

| 구분 | Tool 선택(제안) | Tool 실행 |
|---|---|---|
| 수행 주체 | LLM | 애플리케이션(Python 코드) |
| 결과물 | Tool 이름과 인자(Tool Call) | 실제 반환값 |
| 오류 발생 위치 | 잘못된 Tool/인자를 제안 | 함수 실행 중 예외 발생 |
| 신뢰 가능 여부 | 검증 전에는 신뢰할 수 없음 | 검증과 예외 처리를 통해 신뢰 확보 |

### 3.8 사용 시점과 적용 조건

산술 계산, 파일 조회, 외부 데이터 검색처럼 정확성이 중요하거나 LLM이 직접 접근할 수 없는 자원이 필요한 경우에 Tool을 사용한다.   
단순한 설명이나 요약처럼 LLM의 언어 생성 능력만으로 충분한 경우에는 Tool 없이 직접 응답하는 편이 더 빠르고 간단하다.  

### 3.9 한계와 주의사항

- LLM이 제안하는 인자는 자료형이 맞지 않거나 필수 인자가 누락될 수 있으므로 실행 전에 반드시 검증해야 한다.  
- `calculate()`처럼 임의의 문자열을 계산에 사용하는 Tool은 `eval()`을 그대로 쓰면 임의 코드 실행으로 이어질 수 있다. 
- 이 실습의 `calculate()`는 `ast` 모듈로 사칙연산만 허용하도록 제한되어 있다.  
- Tool 호출이 많아질수록 LLM 호출 횟수와 지연시간이 늘어나므로, 꼭 필요한 경우에만 Tool을 등록해야 한다.  

### 3.10 자주 발생하는 오해

"LLM이 Tool을 직접 실행한다"는 것은 흔한 오해이다.  
실제로 LLM은 어떤 Tool을 어떤 인자로 호출하면 좋을지 **제안**만 하고,  
실제 Python 함수를 호출하는 것은 애플리케이션 코드이다.  
이 구분이 없으면 Tool 실행 중 예외가 발생했을 때 원인을 LLM 쪽에서 찾으려는 실수를 하게 된다.  

## 4. 실행 구조

```mermaid
flowchart TD
    A[사용자 요청] --> B["model_with_tools.invoke 호출<br/>AIMessage tool_calls 반환"]
    B --> C{tool_calls 있음?}
    C -- 없음 --> G1[AIMessage content를<br/> 최종 응답으로 사용]
    C -- 있음 --> D["tool_calls 개수만큼 반복:<br/>execute_tool_call<br/> → ToolMessage 포장"]
    D --> F["model_with_tools.invoke 재호출 1회<br/>요청 + AIMessage<br/> + 모든 ToolMessage"]
    F --> G2[최종 응답 생성]
```

## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [1]:
import json

from agentic_ai.config import get_settings
from agentic_ai.logging_utils import save_log
from agentic_ai.models import get_chat_model
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.paths import DATA_DIR, OUTPUT_DIR, PROJECT_ROOT
from agentic_ai.tools import calculate, check_required_fields, load_document, search_keyword

settings = get_settings()
print_environment_summary(settings, needs_chat_model=True)


[환경 설정 확인]
- 프로젝트: C:\Users\magpi\agentic_ai_lab_202607
- 데이터: C:\Users\magpi\agentic_ai_lab_202607\data
- 출력: C:\Users\magpi\agentic_ai_lab_202607\outputs
- OPENAI_API_KEY: 설정됨
- Chat Model: gpt-4.1-mini


## 6. 최소 실행 예제

Tool 없이 LLM을 한 번 호출해 기본 파이프라인을 확인한다.

In [2]:
model = get_chat_model()
response = model.invoke("Tool Calling이 뭔지 한 문장으로 설명해줘.")
print(response.content)

Tool Calling은 인공지능이 외부 도구나 API를 호출해 추가 정보를 얻거나 작업을 수행하는 기능입니다.


## 7. 단계별 구현

### 7.1 Python 함수 정의

`calculate`, `search_keyword`, `load_document`, `check_required_fields`는 `src/agentic_ai/tools.py`에 일반 Python 함수로 정의되어 있다.  
Python 함수 자체는 모델에 이름·설명·인자 Schema가 노출되지 않았으므로 아직 Tool이 아니다. 
다음 단계에서 `@tool`로 감싸 함수형 Tool로 변환한다.  

In [3]:
print(calculate("25 + 17"))
print(search_keyword("리모트워크 만족도는 74%였다.", "만족도"))

42
{'keyword': '만족도', 'found': True, 'position': 6}


### 7.2 Tool Schema 작성

`@tool` 데코레이터로 Python 함수를 감싸면 함수의 시그니처와 Docstring이 자동으로
Tool Schema(이름, 설명, 인자 자료형)로 변환된다.

In [4]:
from langchain_core.tools import tool


# @tool은 함수 시그니처와 Docstring을 모델이 읽을 수 있는 Tool Schema로 바꾼다.
@tool
def calculate_tool(expression: str) -> float:
    """사칙연산 수식을 계산한다. 예: '25 + 17'"""
    return calculate(expression)


@tool
def search_keyword_tool(text: str, keyword: str) -> dict:
    """텍스트 안에서 특정 키워드를 검색하고 위치를 반환한다."""
    return search_keyword(text, keyword)


@tool
def load_document_tool(file_path: str) -> str:
    """지정한 경로의 문서를 읽어서 본문을 반환한다."""
    return load_document(file_path)


@tool
def check_required_fields_tool(data: dict, required: list[str]) -> list[str]:
    """딕셔너리에서 필수 필드 중 누락된 필드 목록을 반환한다."""
    return check_required_fields(data, required)


# 목록은 모델에 공개할 Tool 집합, Registry는 이름으로 실제 함수를 찾기 위한 실행용 매핑이다.
TOOLS = [calculate_tool, search_keyword_tool, load_document_tool, check_required_fields_tool]
TOOL_REGISTRY = {t.name: t for t in TOOLS}

for t in TOOLS:
    print(t.name, "-", t.description)

calculate_tool - 사칙연산 수식을 계산한다. 예: '25 + 17'
search_keyword_tool - 텍스트 안에서 특정 키워드를 검색하고 위치를 반환한다.
load_document_tool - 지정한 경로의 문서를 읽어서 본문을 반환한다.
check_required_fields_tool - 딕셔너리에서 필수 필드 중 누락된 필드 목록을 반환한다.


### 7.3 LLM에 Tool 등록

`bind_tools()`로 LLM에 사용 가능한 Tool 목록을 알려준다.

In [5]:
# bind_tools는 Tool 정보를 모델에 알려줄 뿐, 모델이 Python 함수를 직접 실행하게 하지는 않는다.
model_with_tools = model.bind_tools(TOOLS)

# 첫 응답의 tool_calls에는 모델이 제안한 함수 이름과 인자만 담긴다.
ai_message = model_with_tools.invoke("25와 17을 더한 값을 계산해줘.")
print("content:", ai_message.content)
print("tool_calls:", ai_message.tool_calls)

content: 
tool_calls: [{'name': 'calculate_tool', 'args': {'expression': '25 + 17'}, 'id': 'call_3VfeqigFBXOgiVgILZsbtdTq', 'type': 'tool_call'}]


### 7.4 Tool 호출 여부 확인

`ai_message.tool_calls`가 비어 있으면 LLM이 Tool 없이 직접 답할 수 있다고 판단한
것이고, 값이 있으면 어떤 Tool을 어떤 인자로 호출하고 싶은지를 담고 있다.

In [6]:
if ai_message.tool_calls:
    for call in ai_message.tool_calls:
        print(f"Tool 이름: {call['name']}, 인자: {call['args']}")
else:
    print("Tool 호출이 제안되지 않았습니다. 직접 응답:", ai_message.content)

Tool 이름: calculate_tool, 인자: {'expression': '25 + 17'}


### 7.5~7.6 인자 파싱과 Python 함수 실행

**TODO**: `execute_tool_call()`을 작성한다.

- `call["name"]`이 `TOOL_REGISTRY`에 없으면 `{"status": "error", "error": "알 수 없는 Tool: ..."}`을 반환한다.
- 있으면 `TOOL_REGISTRY[name].invoke(call["args"])`를 실행한다.
- 실행 중 예외(`TypeError`, `ValueError`, `FileNotFoundError`)가 발생하면
  `{"status": "error", "error": str(exc)}`를 반환한다.
- 성공하면 `{"status": "ok", "output": 결과값}`을 반환한다.

예상 출력: `{"status": "ok", "output": 42}` 또는 `{"status": "error", "error": "..."}`

In [7]:
def execute_tool_call(call: dict) -> dict:
    """Tool Call을 검증하고 실제 Python 함수를 실행한다."""
    name = call["name"]
    # 모델이 제안한 이름을 신뢰하지 않고 허용된 Registry에서 먼저 확인한다.
    if name not in TOOL_REGISTRY:
        return {"status": "error", "error": f"알 수 없는 Tool: {name}"}

    tool_fn = TOOL_REGISTRY[name]
    try:
        # invoke에 args 딕셔너리를 넘기면 Tool Schema에 맞춰 인자를 검증한 뒤 실행한다.
        # 인자 검증 실패 시 Pydantic의 ValidationError가 발생하는데, 이는 ValueError의
        # 하위 클래스이므로 아래 except 절에서 함께 잡힌다(9절 broken_calls의 c2, c3 참고).
        output = tool_fn.invoke(call["args"])
        return {"status": "ok", "output": output}
    except (TypeError, ValueError, FileNotFoundError) as exc:
        return {"status": "error", "error": str(exc)}


sample_result = execute_tool_call(ai_message.tool_calls[0])
print(sample_result)

{'status': 'ok', 'output': 42}


### 7.7~7.8 Tool 결과 반환과 최종 응답 생성

Tool 실행 결과를 `ToolMessage`로 감싸서 원래 대화에 이어붙이고, LLM을 다시 호출해
최종 응답을 생성한다.

In [8]:
from langchain_core.messages import HumanMessage, ToolMessage


def run_tool_calling(user_request: str) -> dict:
    """Tool Calling 전체 파이프라인을 실행하고 로그를 반환한다."""
    ai_message = model_with_tools.invoke(user_request)
    log = {"user_request": user_request, "tool_calls": ai_message.tool_calls, "tool_results": []}

    if not ai_message.tool_calls:
        log["final_answer"] = ai_message.content
        return log

    # 각 실행 결과를 원래 call의 id와 연결해야 모델이 어떤 요청의 결과인지 알 수 있다.
    tool_messages = []
    for call in ai_message.tool_calls:
        result = execute_tool_call(call)
        log["tool_results"].append({"name": call["name"], "args": call["args"], **result})
        tool_messages.append(ToolMessage(content=json.dumps(result, ensure_ascii=False), tool_call_id=call["id"]))

    # 사용자 요청, Tool Call, Tool 결과를 함께 전달해 최종 자연어 답변을 생성한다.
    final_message = model_with_tools.invoke(
        [HumanMessage(content=user_request), ai_message, *tool_messages]
    )
    log["final_answer"] = final_message.content
    return log


calc_log = run_tool_calling("25와 17을 더한 값을 계산해줘.")
print(calc_log)

{'user_request': '25와 17을 더한 값을 계산해줘.', 'tool_calls': [{'name': 'calculate_tool', 'args': {'expression': '25 + 17'}, 'id': 'call_S9jv4pnsuUw9Z6bBJZMvEeGQ', 'type': 'tool_call'}], 'tool_results': [{'name': 'calculate_tool', 'args': {'expression': '25 + 17'}, 'status': 'ok', 'output': 42}], 'final_answer': '25와 17을 더한 값은 42입니다.'}


## 8. 실행 결과 관찰

여러 요청으로 실행해서 어떤 Tool이 선택되고, 어떤 인자가 전달되고, 실행 결과가
무엇인지 확인한다.

In [9]:
requests = [
    "128을 4로 나눠줘.",
    "다음 텍스트에서 '만족도'라는 단어를 찾아줘: 리모트워크 만족도는 74%였다.",
]

for req in requests:
    result_log = run_tool_calling(req)
    print(f"\n요청: {req}")
    print("선택된 Tool:", [c["name"] for c in result_log["tool_calls"]])
    print("Tool 결과:", result_log["tool_results"])
    print("최종 응답:", result_log["final_answer"])


요청: 128을 4로 나눠줘.
선택된 Tool: ['calculate_tool']
Tool 결과: [{'name': 'calculate_tool', 'args': {'expression': '128 / 4'}, 'status': 'ok', 'output': 32.0}]
최종 응답: 128을 4로 나누면 32입니다.

요청: 다음 텍스트에서 '만족도'라는 단어를 찾아줘: 리모트워크 만족도는 74%였다.
선택된 Tool: ['search_keyword_tool']
Tool 결과: [{'name': 'search_keyword_tool', 'args': {'text': '리모트워크 만족도는 74%였다.', 'keyword': '만족도'}, 'status': 'ok', 'output': {'keyword': '만족도', 'found': True, 'position': 6}}]
최종 응답: 텍스트에서 '만족도'라는 단어는 6번째 위치에서 발견되었습니다.


**결과 해석**:  
`tool_calls`에는 LLM이 제안한 Tool 이름과 인자가 담기고, `tool_results`에는 애플리케이션이 실제로 실행한 결과가 담긴다.  
두 값이 함께 기록되어야 어떤 Tool이 선택되었고 실제로 어떤 값이 계산되었는지를 구분해서 확인할 수 있다.  

## 9. 실패 실험: 잘못된 Tool Call

실제 LLM 호출 없이, 다섯 가지 오류 상황을 직접 만든 Tool Call로 재현한다.   
이렇게 하면 LLM의 비결정성과 무관하게 오류 처리 로직을 안정적으로 검증할 수 있다.  

In [10]:
broken_calls = [
    {"name": "unknown_tool", "args": {}, "id": "c1"},                       # 존재하지 않는 Tool
    {"name": "calculate_tool", "args": {}, "id": "c2"},                     # 누락된 인자
    {"name": "calculate_tool", "args": {"expression": ["25", "+", "17"]}, "id": "c3"},  # 자료형 오류
    {"name": "calculate_tool", "args": {"expression": "1/0"}, "id": "c4"},  # 실행 중 예외
    {"name": "load_document_tool", "args": {"file_path": "no_such_file.txt"}, "id": "c5"},  # 존재하지 않는 파일
]

for call in broken_calls:
    result = execute_tool_call(call)
    print(f"{call['name']} / {call['args']} -> {result}")

unknown_tool / {} -> {'status': 'error', 'error': '알 수 없는 Tool: unknown_tool'}
calculate_tool / {} -> {'status': 'error', 'error': '1 validation error for calculate_tool\nexpression\n  Field required [type=missing, input_value={}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing'}
calculate_tool / {'expression': ['25', '+', '17']} -> {'status': 'error', 'error': "1 validation error for calculate_tool\nexpression\n  Input should be a valid string [type=string_type, input_value=['25', '+', '17'], input_type=list]\n    For further information visit https://errors.pydantic.dev/2.13/v/string_type"}
calculate_tool / {'expression': '1/0'} -> {'status': 'error', 'error': '0으로 나눌 수 없습니다.'}
load_document_tool / {'file_path': 'no_such_file.txt'} -> {'status': 'error', 'error': '문서를 찾을 수 없습니다: no_such_file.txt'}


**원인 분석 질문**

- 각 사례에서 `status`가 `"error"`가 되는 원인은 서로 같은가, 다른가?
- `execute_tool_call()`이 이 예외들을 잡지 못했다면 `run_tool_calling()` 전체가
  어떻게 되었을까?
- 존재하지 않는 Tool 이름은 어느 단계(LLM의 제안 단계 vs 애플리케이션의 실행 단계)에서
  걸러지는가?

## 10. 오류 수정 실습

LLM은 "10을 0으로 나눠줘"를 Tool 없이 직접 거절할 수도 있다.  
그러면 Tool 실행 실패를 검증할 수 없으므로, 잘못된 Tool Call을 직접 주입하고 로그에서 실패 Tool을 추출한다.  

**TODO**: 
`summarize_tool_failures()`를 작성한다. `status == "error"`인 `tool_results`의 Tool 이름만 반환해야 한다.  


In [11]:
def summarize_tool_failures(log: dict) -> list[str]:
    """실행 로그에서 실패한 Tool 이름 목록을 추출한다."""
    return [
        result["name"]
        for result in log.get("tool_results", [])
        if result.get("status") == "error"
    ]


# LLM이 0으로 나누기를 직접 거절할 수도 있으므로 Tool 실패는 결정적으로 주입한다.
forced_call = {"name": "calculate_tool", "args": {"expression": "10 / 0"}, "id": "forced-1"}
forced_result = execute_tool_call(forced_call)
division_error_log = {
    "user_request": "10을 0으로 나눠줘.",
    "tool_calls": [forced_call],
    "tool_results": [{"name": forced_call["name"], "args": forced_call["args"], **forced_result}],
    "final_answer": f"Tool 실행 실패: {forced_result['error']}",
}
print("실패한 Tool:", summarize_tool_failures(division_error_log))
print("최종 응답:", division_error_log["final_answer"])


실패한 Tool: ['calculate_tool']
최종 응답: Tool 실행 실패: 0으로 나눌 수 없습니다.


**수정 결과 재검증**

In [12]:
assert summarize_tool_failures(
    {"tool_results": [{"name": "calculate_tool", "status": "error"}]}
) == ["calculate_tool"]
assert summarize_tool_failures({"tool_results": [{"name": "calculate_tool", "status": "ok"}]}) == []
print("재검증 통과")

재검증 통과


## 11. 도전 과제

1. `search_keyword_tool`에 존재하지 않는 키워드를 검색하도록 요청해서 `found: False`
   결과가 최종 응답에 어떻게 반영되는지 확인한다.
2. 새로운 Tool(예: `count_words_tool`)을 하나 더 만들어 `TOOLS` 목록에 등록하고,
   LLM이 이를 선택하는지 관찰한다.
3. 한 번의 요청에서 두 개 이상의 Tool이 동시에 호출되는 입력을 만들어본다.

## 12. 테스트

**테스트 유형: 단위 테스트 — 결정적, 외부 API 호출 없음**

LLM 호출 없이 `execute_tool_call()`의 오류 처리 로직을 검증한다.

In [13]:
assert execute_tool_call({"name": "unknown_tool", "args": {}, "id": "t1"})["status"] == "error"
assert execute_tool_call({"name": "calculate_tool", "args": {"expression": "1/0"}, "id": "t2"})["status"] == "error"
assert execute_tool_call({"name": "calculate_tool", "args": {"expression": "25 + 17"}, "id": "t3"}) == {
    "status": "ok", "output": 42
}
print("execute_tool_call 테스트 통과")

execute_tool_call 테스트 통과


## 13. 결과 저장

In [14]:
tool_calling_log = {
    "calc_example": calc_log,
    "requests_observed": requests,
    "broken_calls_tested": [c["name"] for c in broken_calls],
    "division_error_log": division_error_log,
}
saved_path = save_log(tool_calling_log, OUTPUT_DIR / "logs" / "01-3_tool_calling_log.json")
print("저장 위치:", saved_path)

저장 위치: C:\Users\magpi\agentic_ai_lab_202607\outputs\logs\01-3_tool_calling_log.json


## 14. 핵심 정리

- Tool Calling은 함수형 Tool뿐 아니라 검색·파일·MCP 등 다양한 도구 사용을 포괄한다.
- Function Calling은 이 실습처럼 개발자가 정의한 함수를 Tool로 사용하는 경우를 가리킨다.
- Tool Schema는 Python 함수를 LLM이 이해할 수 있는 이름·설명·인자 형식으로 변환한 것이다.
- LLM은 Tool 호출을 제안(Tool Call)할 뿐이며, 실제 함수 실행은 애플리케이션이 담당한다.
- Tool 실행 결과는 Tool Message로 LLM에 다시 전달되어야 최종 응답에 반영된다.
- 존재하지 않는 Tool, 누락된 인자, 자료형 오류, 실행 중 예외는 모두 애플리케이션
  단계에서 감지하고 처리해야 한다.

## 15. 확인 문제

1. Function Calling과 Tool Calling의 관계를 설명하시오.
2. "LLM이 Tool을 실행한다"는 표현이 왜 정확하지 않은지 설명하시오.
3. Tool Call과 Tool Message의 차이를 설명하시오.
4. `calculate()`가 `eval()` 대신 `ast` 모듈을 사용하는 이유는 무엇인가?
5. 이번 실습의 실패 실험에서 다룬 다섯 가지 오류 유형을 나열하시오.